***

Preparing Workspace

***

In [ ]:


export=False


from pathlib import Path
import pandas as pd
import time
import sys

PATH_GIT = Path.home() / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen'
PATH_CODE    = PATH_GIT / 'Data' / 'RITIS'
PATH_CONFIG0 = PATH_GIT / 'config'
PATH_CONFIG  = PATH_CODE / 'config'
PATH_SQL = PATH_GIT / 'Data' / 'RITIS' / 'sql_scripts'

sys.path.append(str(PATH_CONFIG0))
import functions as func

pd.set_option('display.max_columns', None)


# SharePoint
PATH_SP = Path.home() / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents' 
PATH_CONGESTION = PATH_SP / 'Data' / 'Safe Equitable Resilient Infrastructure' / 'Congestion'
PATH_PHED  = PATH_CONGESTION / 'RITIS' / 'PHED'
PATH_LOTTR = PATH_CONGESTION / 'RITIS' / 'LOTTR'
PATH_SERVER = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data")



***

Congestion_2

***

In [ ]:
# Keep track of time amounted while SQL query is running through all years
start_time = time.time()

# Set up parameters for SQL queries
db = 'NPMRDS'
year_start = 2017
year_end   = 2024
years_to_import = range(year_start, year_end+1)
list_df_sql = []

# Loop through years to calculate % road miles congested by year
for year in years_to_import:

    print()
    print(f'Running congestion calculator on year {year}...')
    print()

    with open(PATH_SQL / f'Congestion_2_{year}_pct_rd_miles_congested.sql', 'r') as query:
        query_string = query.read()
    
    df_sql = func.sqlqry_to_df(query_string, db)
    df_sql['Year'] = year
    list_df_sql.append(df_sql)
    print()

# Calculate time
print()
print('Finished!!')
print(f"Process complete.  It took --- {round((time.time() - start_time)/60, 1)} minutes ---")
print()


# Concatenate all years together
df_congestion2 = pd.concat(list_df_sql)
df_congestion2['MPO'] = 'SACOG'
df_congestion2 = df_congestion2.set_index(['MPO', 'Year']).reset_index()
df_congestion2 = df_congestion2.sort_values('Year', ascending=False).reset_index(drop=True)
display(df_congestion2)

In [ ]:

if export:
    indicator = 'Congestion_2'
    sample_type = 'RITIS'
    geography = 'MPO'
    df_about = func.write_about(sample_type, indicator, year_start, year_end, geography=geography)
    files_out = [PATH_CONGESTION / indicator / f"{indicator} UZA RITIS.xlsx", PATH_SERVER / f"{indicator} UZA RITIS.xlsx"]
    for file_out in files_out:
        with pd.ExcelWriter(file_out, engine='xlsxwriter') as writer:
            df_about      .to_excel(writer, index=False, sheet_name='About', header=False)
            df_congestion2.to_excel(writer, index=False, sheet_name='SACOG'              )